## **5.4 Session Agenda**

We test ***LLM's response by passing wrong inputs*** at *3 separate levels*:

1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

# ==============================================================================

# **INTERVIEW QUESTION 1: How LLM Calls the Tools?**

**NOTE: LLM does not execute Python code**.

# **1. Install Required Libraries**

In [3]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

GENERATION_MODEL = "gpt-4.1-mini"

print("OpenAI client configured successfully.")
print("Model:", GENERATION_MODEL)


OpenAI client configured successfully.
Model: gpt-4.1-mini


# ==============================================================================

# **2.1 Python Application Functions**


In [4]:
def employee_lookup(employee_id):
    employees = {101: {"name": "Madhavan", "department": "AI Research Team"},
                 102: {"name": "Rajesh", "department": "Operations"},
                 103: {"name": "Jagadeeshwari", "department": "AI Engineering Team"}}
    return employees.get(employee_id,{"error": "Employee Not Found"})

def calculate_leave_balance(total_leaves, leaves_taken):
    return {"remaining_leaves": total_leaves - leaves_taken}

def calculate(expression):
    allowed = set("0123456789+-*/(). %")

    if not expression or any(char not in allowed for char in expression):
        return {"error": "Only basic arithmetic expressions are allowed."}
    try:
        return {"result": eval(expression, {"__builtins__": {}}, {})}
    except Exception:
        return {"error": "Invalid arithmetic expression."}

def send_email(to, subject, body):
    return {"status": "simulated","message": f"Email prepared for {to} with subject '{subject}'." }

def book_meeting(title, attendee, time):
    return {"status": "simulated","message": f"Meeting '{title}' prepared with {attendee} at {time}."}

# **2.2 LLM input - expects Tools**

In [8]:
TOOLS = [
    {
        "type": "function",
        "name": "employee_lookup",
        "description": "Look up an employee's name and department using an employee ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "employee_id": {"type": "integer", "description": "A numeric employee ID, such as 101, 102, or 103. "
                        "Never guess or invent an employee ID. "
                        "If the user provides a non-numeric ID, "
                        "ask for clarification instead of calling this tool."}                     
            },
            "required": ["employee_id"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate_leave_balance",
        "description": "Calculate remaining leave from total and taken leaves.",
        "parameters": {
            "type": "object",
            "properties": {
                "total_leaves": {"type": "integer"},
                "leaves_taken": {"type": "integer"}
            },
            "required": ["total_leaves", "leaves_taken"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "Perform a basic arithmetic calculation.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "send_email",
        "description": "Prepare a simulated email.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "book_meeting",
        "description": "Prepare a simulated meeting booking.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "attendee": {"type": "string"},
                "time": {"type": "string"}
            },
            "required": ["title", "attendee", "time"],
            "additionalProperties": False
        }
    }
]

print("Available tools:", [tool["name"] for tool in TOOLS])


Available tools: ['employee_lookup', 'calculate_leave_balance', 'calculate', 'send_email', 'book_meeting']


## ==============================================================================

# **3. Tool Registry Preparation**
**Why?**


A registry connects the **model's selected tool name** to the ***corresponding Python function.***

In [6]:
TOOL_REGISTRY = {"employee_lookup": employee_lookup,
                 "calculate_leave_balance": calculate_leave_balance,
                 "calculate": calculate,
                 "send_email": send_email,
                 "book_meeting": book_meeting}

# **4. Tool Selection**

The model can select the appropriate tool based on the user's request.

## ***4.1 Without Tools***

In [ ]:
question = "Who is employee 102?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question)
print(response.output_text)

## ***4.2 With Tool***

In [ ]:
question = "Who is employee 102?"

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                   tools=TOOLS)
print("-"*30)
print(response.output)
print("-"*30)

for item in response.output:
    if item.type == "function_call":
        print("Selected tool:", item.name)
        print("Arguments:", item.arguments)

# ==============================================================================

# **MAIN PART of this Module 5.4**

We test ***LLM's response by passing wrong inputs*** at *3 separate levels*:

1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

# **Level 1: User Prompt Level**
- *the user's request itself is incomplete or invalid.*

### **1. Employee ID with wrong type**

In [9]:
question = "Find employee AB and tell me their name and department."

response = client.responses.create(model=GENERATION_MODEL,
                                   input=question,
                                   tools=TOOLS)

print("-"*30)
print(response.output)
print("-"*30)

for item in response.output:
    if item.type == "function_call":
        print("Selected tool:", item.name)
        print("Arguments:", item.arguments)
    else:
        print("No tool call was produced.")
        print("Final output:", response.output_text)

------------------------------
[ResponseOutputMessage(id='msg_0435f94261f6e525006ac8c19bcccc87d29d78c648024d0b87', content=[ResponseOutputText(annotations=[], text='Could you please provide the employee ID for employee AB? Employee IDs are usually numeric. This will help me find the correct information.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]
------------------------------
No tool call was produced.
Final output: Could you please provide the employee ID for employee AB? Employee IDs are usually numeric. This will help me find the correct information.


# **Add this step: Define the "Client Response Extractor"**

In [10]:
def client_response_extractor(question):
  response = client.responses.create(model=GENERATION_MODEL,
                                    input=question,
                                    tools=TOOLS)
  print("-"*30)
  print(response.output)
  print("-"*30)

  for item in response.output:
      if item.type == "function_call":
          print("Selected tool:", item.name)
          print("Arguments:", item.arguments)
      else:
          print("No tool call was produced.")
          print("Final output:", response.output_text)

### **2. Unknown employee ID**

In [14]:
question = "Find employee 55 and tell me their name and department."
client_response_extractor(question)

------------------------------
[ResponseFunctionToolCall(arguments='{"employee_id":55}', call_id='call_u8gNPnfiP8q0a3GjpClJjnRg', name='employee_lookup', type='function_call', id='fc_0ea4afd92dd1b962006ac8c25e5c4487d297e6bb552fcc8146', async_=None, caller=None, namespace=None, status='completed')]
------------------------------
Selected tool: employee_lookup
Arguments: {"employee_id":55}


### **3. Missing leave values**

In [ ]:
question = "Calculate the leave balance, but I will not give you total leaves or leaves taken."
client_response_extractor(question)

### **4. Division by zero**

In [ ]:
question = "Calculate 10 divided by zero."
client_response_extractor(question)

### **5. Unsupported calculation**

In [ ]:
question = "Calculate DROP TABLE employees."
client_response_extractor(question)

### **6. Invalid email recipient**

In [ ]:
question = "Prepare an email to not-an-email saying hello."
client_response_extractor(question)

### **7. Missing meeting title**

In [ ]:
question = "Book a meeting with John tomorrow, but I have not provided a meeting title."
client_response_extractor(question)

# ==============================================================================

# **Level 2 & 3: Tool Argument Level & Tool Execution Level Validation**

# **Remainder...**

1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

We will intentionally pass incorrect arguments to each tool.

# **Step 1**: Define the Tool Safety Settings or Tool Safety Wrapper

In [15]:
import inspect
import json

def execute_tool_safely(tool_name, arguments):
    if tool_name not in TOOL_REGISTRY:
        return {"status": "error","error_type": "UnknownTool","error_message": f"Unknown tool requested: {tool_name}"}

    function = TOOL_REGISTRY[tool_name]

    try:
        # Python signature validation: catches missing and unexpected arguments.
        bound = inspect.signature(function).bind(**arguments)

        # Explicit type validation for demonstration.
        if tool_name == "employee_lookup":
            if not isinstance(arguments.get("employee_id"),int):
                raise TypeError("employee_id must be an integer.")

        elif tool_name == "calculate_leave_balance":
            for name in ("total_leaves", "leaves_taken"):
                value = arguments.get(name)
                if not isinstance(value, int):
                    raise TypeError(f"{name} must be an integer.")

        elif tool_name == "calculate":
            if not isinstance(arguments.get("expression"), str):
                raise TypeError("expression must be a string.")

        elif tool_name == "send_email":
            for name in ("to", "subject", "body"):
                if not isinstance(arguments.get(name), str):
                    raise TypeError(f"{name} must be a string.")

            if "@" not in arguments["to"]:
                raise ValueError("Invalid email address.")

        elif tool_name == "book_meeting":
            for name in ("title", "attendee", "time"):
                if not isinstance(arguments.get(name), str):
                    raise TypeError(f"{name} must be a string.")

            if not arguments["title"].strip():
                raise ValueError("Meeting title cannot be empty.")

        result = function(**bound.arguments)
        return {"status": "success","result": result}

    except TypeError as e:
        return {"status": "error","error_type": "TypeError","error_message": str(e)}

    except ValueError as e:
        return {"status": "error","error_type": "ValueError","error_message": str(e)}

    except Exception as e:
        return {"status": "error","error_type": type(e).__name__,"error_message": str(e)}

print("Safe tool execution wrapper ready.")

Safe tool execution wrapper ready.


### **Negative Test 1: employee_lookup — wrong type**

In [16]:
result = execute_tool_safely("employee_lookup", {"employee_id": "ABC"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': 'employee_id must be an integer.'}


### **Negative Test 2: employee_lookup — missing parameter**

In [ ]:
result = execute_tool_safely("employee_lookup", {})

print("Tool Result:")
print(result)

### **Negative Test 3: employee_lookup — unknown employee**

In [17]:
result = execute_tool_safely("employee_lookup", {"employee_id": 999})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'success', 'result': {'error': 'Employee Not Found'}}


### **Negative Test 4: calculate_leave_balance — wrong type**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": "24", "leaves_taken": 7})

print("Tool Result:")
print(result)

In [ ]:
result = execute_tool_safely("calculate_leave_balance", 24)

print("Tool Result:")
print(result)

### **Negative Test 5: calculate_leave_balance — missing parameter**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24})

print("Tool Result:")
print(result)

### **Negative Test 6: calculate_leave_balance — unexpected parameter**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": 7, "extra": 1})

print("Tool Result:")
print(result)

### **Negative Test 7: calculate_leave_balance — negative leaves taken**

In [ ]:
result = execute_tool_safely("calculate_leave_balance", {"total_leaves": 24, "leaves_taken": -5})

print("Tool Result:")
print(result)

### **Negative Test 8: calculate — division by zero**

In [ ]:
result = execute_tool_safely("calculate", {"expression": "10 / 0"})

print("Tool Result:")
print(result)

### **Negative Test 9: calculate — unsupported expression**

In [ ]:
result = execute_tool_safely("calculate", {"expression": "DROP TABLE employees"})

print("Tool Result:")
print(result)

### **Negative Test 10: calculate — wrong type**

In [ ]:
result = execute_tool_safely("calculate", {"expression": 125})

print("Tool Result:")
print(result)

### **Negative Test 11: calculate — empty expression**

In [ ]:
result = execute_tool_safely("calculate", {"expression": ""})

print("Tool Result:")
print(result)

### **Negative Test 12: send_email — invalid email**

In [18]:
result = execute_tool_safely("send_email", {"to": "not-an-email", "subject": "Test", "body": "Hello"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'ValueError', 'error_message': 'Invalid email address.'}


### **Negative Test 13: send_email — missing body**

In [19]:
result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": "Test"})

print("Tool Result:")
print(result)

Tool Result:
{'status': 'error', 'error_type': 'TypeError', 'error_message': "missing a required argument: 'body'"}


### **Negative Test 14: send_email — wrong subject type**

In [ ]:
result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": 123, "body": "Hello"})

print("Tool Result:")
print(result)

### **Negative Test 15: send_email — unexpected parameter**

In [ ]:
result = execute_tool_safely("send_email", {"to": "hr@company.com", "subject": "Test", "body": "Hello", "cc": "x@company.com"})

print("Tool Result:")
print(result)

### **Negative Test 16: book_meeting — empty title**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "", "attendee": "John", "time": "10:00 AM"})

print("Tool Result:")
print(result)

### **Negative Test 17: book_meeting — wrong attendee type**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": 123, "time": "10:00 AM"})

print("Tool Result:")
print(result)

### **Negative Test 18: book_meeting — missing time**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": "John"})

print("Tool Result:")
print(result)

### **Negative Test 19: book_meeting — unexpected parameter**

In [ ]:
result = execute_tool_safely("book_meeting", {"title": "Review", "attendee": "John", "time": "10:00 AM", "location": "Bangalore"})

print("Tool Result:")
print(result)

# ============================================================================

# **5. The Agent Loop - 2 IMPORTANT POINTS**


1. A single tool call handles one decision.
2. An **agent loop** continues the cycle when the model needs additional tool calls.

# **INTERVIEW QUESTION 2:**
### **How Agent works with LLM Call when you have multiple tools?**

**User → Model → Tool → Result → Model → Tool → Result → Model → Final Answer**

# ==============================================================================

# **6. Run the Agent**

## **Define the Agent Capabilities - 10 Steps**

1. Receive User Input
2. Store Conversation
3. Start Agent Loop
4. Send Request to LLM
5. Check for Tool Call
6. Save LLM Response
7. Execute Selected Tool
8. Get Tool Result
9. Send Tool Result Back to LLM
10. Repeat Until Final Answer or Maximum Steps Reached

In [23]:
def serialize_response_item(item):
    """Convert an SDK response item into API-compatible input data."""

    if hasattr(item, "model_dump"):
        item = item.model_dump(
            exclude_none=True,
            by_alias=True
        )

    if isinstance(item, dict):
        result = {}

        for key, value in item.items():
            if value is None:
                continue

            # Normalize the Python SDK field name if it remains.
            if key == "async_":
                key = "async"

            result[key] = serialize_value(value)

        return result

    return serialize_value(item)


def serialize_value(value):
    """Recursively normalize nested dictionaries and lists."""

    if isinstance(value, dict):
        return serialize_response_item(value)

    if isinstance(value, list):
        return [serialize_value(item) for item in value]

    return value

In [24]:
import json

def run_agent_with_error_handling(user_input, max_steps=5):
    input_items = [{"role": "user", "content": user_input}]

    for step in range(1, max_steps + 1):
        response = client.responses.create(model=GENERATION_MODEL,
                                           input=input_items,
                                           tools=TOOLS )

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            return response.output_text

      #  input_items.extend([item.model_dump() for item in response.output]) #--> part 1 work is done

        #input_items.extend([item.model_dump(exclude_none=True, by_alias=True) for item in response.output])


      # Preserve ALL response output items in conversation history.
        input_items.extend([    serialize_response_item(item)    for item in response.output     ])


        for tool_call in tool_calls:
            try:
                arguments = json.loads(tool_call.arguments)
                tool_result = execute_tool_safely(tool_call.name, arguments)
            except json.JSONDecodeError as e:
                tool_result = {"status": "error","error_type": "InvalidJSON","error_message": f"Could not parse tool arguments: {e}"}

            print(f"Step {step}")
            print("Tool:", tool_call.name)
            print("Arguments:", arguments)
            print("Result:", tool_result)

            input_items.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(tool_result)
            })

    return "Maximum agent steps reached."

### **Single-step task**

In [ ]:
run_agent_with_error_handling("Find employee 102 and tell me their name and department.")

### **Multi-step task**

In [25]:
run_agent_with_error_handling("""Find employee 103. Then prepare a simulated email to HR that mentions the employee's name and
                               department and asks HR to confirm the employee's leave balance.""")

Step 1
Tool: employee_lookup
Arguments: {'employee_id': 103}
Result: {'status': 'success', 'result': {'name': 'Jagadeeshwari', 'department': 'AI Engineering Team'}}
Step 2
Tool: send_email
Arguments: {'to': 'hr@company.com', 'subject': 'Leave Balance Confirmation for Jagadeeshwari', 'body': 'Dear HR Team,\n\nI hope this message finds you well. Could you please confirm the leave balance for Jagadeeshwari from the AI Engineering Team?\n\nThank you for your assistance.\n\nBest regards,\n[Your Name]'}
Result: {'status': 'success', 'result': {'status': 'simulated', 'message': "Email prepared for hr@company.com with subject 'Leave Balance Confirmation for Jagadeeshwari'."}}


"The employee with ID 103 is Jagadeeshwari from the AI Engineering Team. I have prepared a simulated email to HR requesting confirmation of Jagadeeshwari's leave balance. If you want, I can assist you further with this."

# ==============================================================================

# **7. FINAL VALIDATION: LLMs Response Vs AI Agent**

### ***The Difference:***
Now we let the **agent loop** handle each problematic request.

In [ ]:
user_input = 'Find employee 999 and tell me their name and department.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

In [ ]:
user_input = 'Calculate 10 / 0.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

In [ ]:
user_input = 'Calculate DROP TABLE employees.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

In [27]:
user_input = 'Prepare an email to not-an-email saying hello.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

USER INPUT: Prepare an email to not-an-email saying hello.
----------------------------------------------------------------------
Step 1
Tool: send_email
Arguments: {'to': 'not-an-email', 'subject': 'Hello', 'body': 'Hello'}
Result: {'status': 'error', 'error_type': 'ValueError', 'error_message': 'Invalid email address.'}
----------------------------------------------------------------------
FINAL ANSWER: It seems "not-an-email" is not a valid email address. Could you please provide a correct email address for sending the email?


In [ ]:
user_input = 'Book a meeting with John but leave the title empty.'

print("USER INPUT:", user_input)
print("-" * 70)

final_answer = run_agent_with_error_handling(user_input)

print("-" * 70)
print("FINAL ANSWER:", final_answer)

# **The BOOM...Finally the AI Agent can determine...**

1. **What the user wants**
2. **Which tool is required**
3. **What arguments are required**
4. **Whether another tool is needed**
5. **When the task is complete**

This is the foundation of an **AI Agent**.

# **UPNEXT...?**

**LangChain**

# ==============================================================================

# **Other 3 INTERVIEW QUESTIONS**

### **1. What is the difference between RAG and Tool Calling?**


**RAG** retrieves knowledge. **Tool Calling** invokes a capability or accesses an external system.

### **2. Does the LLM execute the Python function?**

No.
1. The model requests the **function call**.
2. The application **executes the function** and ***returns the result.***

### **3. What makes a system an Agent?**

The system can decide what
1. action to take,
2. execute tools,
3. observe results and
4. continue until the task is complete.

# **Overall Summary**

We tested wrong inputs at three separate levels:

1. **User Prompt Level** — the user's request itself is incomplete or invalid.
2. **Tool Argument Level** — the arguments passed to a function are missing, unexpected, or the wrong type.
3. **Tool Execution Level** — the arguments are structurally valid but the function returns an error such as `Employee Not Found` or `Invalid arithmetic expression`.

# **THE END.**

In [26]:
for tool in TOOLS:
    if tool.get("name") == "send_email":
        print(tool)

{'type': 'function', 'name': 'send_email', 'description': 'Prepare a simulated email.', 'parameters': {'type': 'object', 'properties': {'to': {'type': 'string'}, 'subject': {'type': 'string'}, 'body': {'type': 'string'}}, 'required': ['to', 'subject', 'body'], 'additionalProperties': False}}
